Created 7/7/2022

want to understand the credit assignment problem - try to see if we have 8 stripes - how many stripes are used? and is the learning easier with the chunk or the non-chunk 


#we want to check -
1) how many stripes hold non-zero values
2) do the stripes correspond with mnt values
3) are there repeats. 

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
sns.set()
from decodedoutputdefs import Precision, training_curve_indv, training_curve_average #class with the compiled definitions

In [15]:
def getfiles(base):
    files = os.listdir(base)
    files = [f for f in files if 'Base.csv' in f]
    full_files = [base+f for f in files if 'EpcLog' not in f]
    print("all files: {}".format(len(full_files)))
    return full_files

def getlcfiles(base):
    files = os.listdir(base)
    files = [f for f in files if '.csv' in f]
    full_files = [base+f for f in files if 'EpcLog' in f]
    return full_files
    

def geterror(base, good_models = []):
    full_files = getfiles(base)
    if len(good_models) > 0: 
        new_full_files = []
        for f in range(len(full_files)):
            model_num = full_files[f].split('model')[1].split('_')[0]
            if "model"+model_num+".csv" in good_models:
                       new_full_files.append(full_files[f])
                       
    full_files = new_full_files
    
    num_models = len(full_files)
    print("plotting files: {}".format(num_models))
    p = Precision()
    L1_means = np.zeros(num_models)
    L1_vars = np.zeros(num_models)
    for f in range(len(full_files)):
        err_model = p.mult_models([full_files[f]])
        L1 = np.abs(err_model)
        L1_means[f]=np.mean(L1)
        L1_vars[f] = np.var(L1)
    #mse = np.sqrt(np.mean(err**2)) 
    total_mean = np.mean(L1_means)
    total_var = np.mean(L1_vars)/num_models
    errorbar_std = np.std(L1_means)/np.sqrt(num_models)
    return err_model, total_mean, total_var, errorbar_std
    
def getlc(base):
    full_files = getlcfiles(base)
    num_models = len(full_files)
    SSE=[]
    Epoch = []
    for f in full_files:
        if 'model0_' in f:
            df = pd.read_csv(f, sep = '\t')
            SSE.append(df['#SSE'])
            Epoch.append(df['|Epoch'])
        else:
            df = pd.read_csv(f,sep = '\t', header = None)
            SSE.append(df[2])
    SSE_np = np.array(SSE)
    SSE_avg = np.mean(SSE_np, axis = 0)
    SSE_std = np.std(SSE_np, axis = 0)/np.sqrt(num_models)
    #plt.plot(Epoch[0],SSE_avg)
    
    return Epoch[0], SSE_avg, SSE_std 
    

def geterror_array(base, good_models = []):
    full_files = getfiles(base)
    if len(good_models) > 0: 
        new_full_files = []
        for f in range(len(full_files)):
            model_num = full_files[f].split('model')[1].split('_')[0]
            if "model"+model_num+".csv" in good_models:
                       new_full_files.append(full_files[f])
    full_files = new_full_files
    num_models = len(full_files)
    print("plotting files: {}".format(num_models))
    p = Precision()
    err = p.mult_models(full_files)
    return err

def plot_title(base):
    
    if 'contexp' in base:
        rewardtype = 'Continuous Exponential'
    elif 'RewThresh' in base:
        rewardtype = 'Binary'
    else:
        rewardtype = 'Continuous Linear'
        
    if 'decodedrew' in base:
        rewardfun = 'Decoded'
    else: 
        rewardfun = 'Unit Difference'
        
    if 'NoStimLoc' in base:
        stimloc = ', No Stim Loc'
    else: 
        stimloc = ", Stim Loc Used"
   
    if 'NoIgnore' in base:
        ignore = ', No Ignore Trials'
    else:
        ignore = ''

    if "StimRange0to45" in base:
        stim_dist = ', StimDist Max 45'
    else:
        stim_dist = ""


    title = 'Reward Function: ' + rewardtype +', ' +rewardfun +stimloc + ignore +stim_dist#+' '+additional_title
    return title


def mse_allmodels(base, base2, experiment, plot_errorbars = True, good_models = []):
    sir2_file = base + "sir2_new/" +base2+experiment
    sir2_chunk_file = base + "sir2_chunk/" +base2+"hybrid/"+experiment
    sir3_file = base + "sir3/" +base2+experiment
    sir3_chunk_file = base + "sir3_chunk/" +base2+"hybrid/"+experiment
    sir4_file = base + "sir4/" +base2+experiment
    sir4_chunk_file = base + "sir4_chunk/" +base2+"hybrid/"+experiment
    
    if len(good_models) > 0:
        print('sir2')
        err_sir2, mse_sir2, var_sir2, std_sir2 = geterror(sir2_file, good_models['sir2'])
        print('sir2_chunk')
        err_sir2_chunk, mse_sir2_chunk, var_sir2_chunk,std_sir2_chunk = geterror(sir2_chunk_file, good_models['sir2_chunk'])
        print('sir3')
        err_sir3, mse_sir3, var_sir3,std_sir3 = geterror(sir3_file, good_models['sir3'])
        print('sir3_chunk')
        err_sir3_chunk, mse_sir3_chunk, var_sir3_chunk,std_sir3_chunk = geterror(sir3_chunk_file, good_models['sir3_chunk'])
        print('sir4')
        err_sir4, mse_sir4, var_sir4,std_sir4 = geterror(sir4_file, good_models['sir4'])
        print('sir4_chunk')
        err_sir4_chunk, mse_sir4_chunk, var_sir4_chunk,std_sir4_chunk = geterror(sir4_chunk_file, good_models['sir4_chunk'])
    else: 
        err_sir2, mse_sir2, var_sir2, std_sir2 = geterror(sir2_file)
        err_sir2_chunk, mse_sir2_chunk, var_sir2_chunk,std_sir2_chunk = geterror(sir2_chunk_file)
        err_sir3, mse_sir3, var_sir3,std_sir3 = geterror(sir3_file, good_models['sir3'])
        err_sir3_chunk, mse_sir3_chunk, var_sir3_chunk,std_sir3_chunk = geterror(sir3_chunk_file)
        err_sir4, mse_sir4, var_sir4,std_sir4 = geterror(sir4_file, good_models['sir4'])
        err_sir4_chunk, mse_sir4_chunk, var_sir4_chunk,std_sir4_chunk = geterror(sir4_chunk_file)
    

    title = plot_title(sir2_file)
    tasks = [2,3,4]
    mse_all_nochunk = [mse_sir2,mse_sir3,mse_sir4]
    std_all_nochunk = [std_sir2,std_sir3, std_sir4]

    mse_all_chunk = [mse_sir2_chunk,mse_sir3_chunk,mse_sir4_chunk]
    std_all_chunk = [std_sir2_chunk,std_sir3_chunk,std_sir4_chunk]

    plt.xlim(1.8,4.2)
    
    if plot_errorbars == True:
        plt.errorbar(tasks,mse_all_nochunk,std_all_nochunk)
        plt.errorbar(tasks,mse_all_chunk, std_all_chunk)
    else:
        plt.plot(tasks,mse_all_nochunk)
        plt.plot(tasks,mse_all_chunk)

        
    plt.xticks(tasks)
    plt.ylabel('Absolute Error Averaged over Models')
    #plt.ylabel(r'$\sqrt{MSE}$')
    plt.xlabel('Number of Items')
    plt.legend(['No Chunk', 'Chunk'], loc = 'lower right')
    plt.title(title)
    
def lc_allmodels(base,base2,experiment, plot_errorbars = True, sir4 = False):
    sir2_file = base + "sir2_new/" +base2+experiment
    sir2_chunk_file = base + "sir2_chunk/" +base2+"hybrid/"+experiment
    sir3_file = base + "sir3/" +base2+experiment
    sir3_chunk_file = base + "sir3_chunk/" +base2+"hybrid/"+experiment
    if sir4:
        sir4_file = base + "sir4/" +base2+experiment
        sir4_chunk_file = base + "sir4_chunk/" +base2+"hybrid/"+experiment
    
    _, SSE_sir2, std_sir2 = getlc(sir2_file)
    _, SSE_sir2_chunk, std_sir2_chunk = getlc(sir2_chunk_file)
    _, SSE_sir3, std_sir3 = getlc(sir3_file)
    Epoch, SSE_sir3_chunk, std_sir3_chunk = getlc(sir3_chunk_file)
    if sir4:
        _, SSE_sir4, std_sir4 = getlc(sir4_file)
        _, SSE_sir4_chunk, std_sir4_chunk = getlc(sir4_chunk_file)
    
    title = plot_title(sir2_file)
    plt.figure()
    if plot_errorbars:
        plt.errorbar(Epoch,SSE_sir2,std_sir2)
        plt.errorbar(Epoch,SSE_sir2_chunk, std_sir2_chunk)

    else:
        plt.plot(Epoch,SSE_sir2)
        plt.plot(Epoch,SSE_sir2_chunk)
    
    plt.title(title+ ', SIR 2 Task')
    plt.xlabel('Epoch')
    plt.ylabel('SSE')
    plt.legend(['No Chunk', 'Chunk'])
    
    plt.figure()
    if plot_errorbars:
        plt.errorbar(Epoch,SSE_sir3, std_sir3)
        plt.errorbar(Epoch,SSE_sir3_chunk, std_sir3_chunk)
    else:
        plt.plot(Epoch,SSE_sir3)
        plt.plot(Epoch,SSE_sir3_chunk)
        
    plt.title(title+ ', SIR 3 Task')
    plt.xlabel('Epoch')
    plt.ylabel('SSE')
    plt.legend(['No Chunk', 'Chunk'])
    
    if sir4:
        plt.figure()
        if plot_errorbars:
            plt.errorbar(Epoch,SSE_sir4, std_sir4)
            plt.errorbar(Epoch,SSE_sir4_chunk, std_sir4_chunk)
        else:
            plt.plot(Epoch,SSE_sir4)
            plt.plot(Epoch,SSE_sir4_chunk)

        plt.title(title+ ', SIR 4 Task')
        plt.xlabel('Epoch')
        plt.ylabel('SSE')
        plt.legend(['No Chunk', 'Chunk'])

def plot_allhist(base,base2,experiment, good_models = [], overlay = False):
    sir2_file = base + "sir2_new/" +base2+experiment
    sir2_chunk_file = base + "sir2_chunk/" +base2+"hybrid/"+experiment
    sir3_file = base + "sir3/" +base2+experiment
    sir3_chunk_file = base + "sir3_chunk/" +base2+"hybrid/"+experiment
    sir4_file = base + "sir4/" +base2+experiment
    sir4_chunk_file = base + "sir4_chunk/" +base2+"hybrid/"+experiment
        
    
    if overlay: 
        title = plot_title(sir2_file)
        
        plt.figure()
        err = geterror_array(sir2_file,good_models['sir2'])       
        plt.hist(err, bins = 50, alpha = 0.6, density = True)        
        err = geterror_array(sir2_chunk_file,good_models['sir2_chunk'])
        plt.hist(err, bins = 50, color = "yellow", alpha = 0.6, density = True)   
        plt.legend(['No Chunk', 'Chunk'])
        plt.title("Task 2, "+title)
        
        plt.figure()
        err = geterror_array(sir3_file,good_models['sir3'])
        plt.hist(err, bins = 50, alpha = 0.6, density = True)
        err = geterror_array(sir3_chunk_file,good_models['sir3_chunk'])
        plt.hist(err, bins = 50, color = 'yellow', alpha = 0.6, density = True)
        plt.legend(['No Chunk', 'Chunk'])
        plt.title("Task 3, "+title)
        
        plt.figure()
        err = geterror_array(sir4_file,good_models['sir4'])
        plt.hist(err, bins = 50, alpha = 0.6, density = True)
        err = geterror_array(sir4_chunk_file,good_models['sir4_chunk'])
        plt.hist(err, bins = 50, color = 'yellow', alpha = 0.6, density = True)
        plt.legend(['No Chunk', 'Chunk'])
        plt.title("Task 4, "+title)
        
        
    else: 
        title = plot_title(sir2_file)
        err = geterror_array(sir2_file,good_models['sir2'])       
        plt.figure()
        plt.hist(err, bins = 50)
        plt.title("Task 2, No Chunk, "+title)

        err = geterror_array(sir2_chunk_file,good_models['sir2_chunk'])
        plt.figure()
        plt.hist(err, bins = 50)
        plt.title("Task 2, Chunk, "+title)

        err = geterror_array(sir3_file,good_models['sir3'])
        plt.figure()
        plt.hist(err, bins = 50)
        plt.title("Task 3, No Chunk, "+title)

        err = geterror_array(sir3_chunk_file,good_models['sir3_chunk'])
        plt.figure()
        plt.hist(err, bins = 50)
        plt.title("Task 3, Chunk, "+title)

        err = geterror_array(sir4_file,good_models['sir4'])
        plt.figure()
        plt.hist(err, bins = 50)
        plt.title("Task 4, No Chunk, "+title)

        err = geterror_array(sir4_chunk_file,good_models['sir4_chunk'])
        plt.figure()
        plt.hist(err, bins = 50)
        plt.title("Task 4, Chunk, "+title)
    
    


In [6]:
#want to see how many stripes are actually being used.

ex_file = 'Z:/leabra/examples/workingmemory/sir4/results/Ring/8Stripe/decodedrew/contNoStimLoc/NoIgnore/sigma12/sir4model0_RewThreshold5.5_decoded_Base.csv'

In [11]:
df = pd.read_csv(ex_file, sep = '\t')

In [12]:
df.head()

,|Run,|Epoch,|Trial,$TrialName,#Err,#TrlDecodedDiff,#SSE,#AvgSSE,#CosDiff,#DA,...,#stripe,#stripe,#stripe,#stripe,#OutDecode,#OutTarget,#InDecode,$Lesion,#LesionProp,$LesionApplied
0,0,-1,0,Store3_152.02823090302758_mnt1_-1_mnt2_-1_mnt3...,0,0.2848,0.0000,0.00000,0.9564,0.0000,...,152.0,0,0,152.0,151.7,152.0,152.0,StimLoc,0,no
1,0,-1,1,Recall4_0_mnt1_-1_mnt2_-1_mnt3_152.02823090302...,1,207.3000,4.3060,0.21530,-0.3113,-0.6063,...,0.0,0,0,152.0,152.7,360.0,0.0,StimLoc,0,no
2,0,-1,2,Store2_217.5335635687211_mnt1_-1_mnt2_217.5335...,0,0.1832,0.0000,0.00000,0.9758,0.0000,...,0.0,0,0,217.8,217.8,217.6,217.6,StimLoc,0,no
3,0,-1,3,Store1_282.7038982090993_mnt1_282.703898209099...,1,26.0700,0.3712,0.01856,0.7718,0.0000,...,0.0,0,0,282.9,256.9,283.0,282.9,StimLoc,0,no
4,0,-1,4,Recall3_152.02823090302758_mnt1_282.7038982090...,1,126.8000,3.5200,0.17600,-0.3038,-0.2648,...,0.0,0,0,0.0,278.8,152.0,0.0,StimLoc,0,no


In [15]:
stripename = []
for key in df.keys():
    if 'stripe' in key:
        stripename.append(key)

In [16]:
stripename

['#stripe',
 '#stripe\x01',
 '#stripe\x02',
 '#stripe\x03',
 '#stripe\x04',
 '#stripe\x05',
 '#stripe\x06',
 '#stripe\x07']

In [18]:
stripes = {}
for s in stripename:
    stripes[s] = df[s]

The first thing we will check is how many of the stripes are actually holding more than just 0s. 

In [24]:
def get_stripes(df):
    stripename = []
    for key in df.keys():
        if 'stripe' in key:
            stripename.append(key)
    stripes = {}
    for s in stripename:
        stripes[s] = df[s]
    return stripes, stripename

def num_stripes_used(stripes,stripename):
    stripes_used = 0
    for i in range(len(stripename)):
        this_stripe = stripes[stripename[i]]
        if np.sum(this_stripe) > 0: 
            stripes_used+=1 
    return stripes_used

def avg_stripes(chunkfile, nochunkfile):
    #sir4_file = base + "sir3/" +base2+experiment
    #sir4_chunk_file = base + "sir3_chunk/" +base2+"hybrid/"+experiment


    #loop through all of these files and get avg stripes
    chunk_files = getfiles(chunkfile)
    chunk_stripes_used = []
    for f in chunk_files:
        stripes,stripename = get_stripes(pd.read_csv(f,sep = '\t'))
        stripes_used = num_stripes_used(stripes,stripename)
        chunk_stripes_used.append(stripes_used)
    
    nochunk_files = getfiles(nochunkfile)
    nochunk_stripes_used = []
    for f in nochunk_files:
        stripes,stripename = get_stripes(pd.read_csv(f,sep = '\t'))
        stripes_used = num_stripes_used(stripes,stripename)
        nochunk_stripes_used.append(stripes_used)
    chunk_avg_stripes = np.mean(chunk_stripes_used)
    nochunk_avg_stripes = np.mean(nochunk_stripes_used)
    
    return chunk_avg_stripes, nochunk_avg_stripes
    

In [82]:
def find_stimuli(t, task): #input should be t = df_recall.iloc[i]['$TrialName']; task tells sir3 or sir4
    #t is a singular trial.
    if task == 2:
        mnt1 = float(t.split('mnt1_')[1].split('_')[0])
        mnt2 = float(t.split('mnt2_')[1].split('_')[0])
        return [mnt1,mnt2]
    if task == 3:
        mnt1 = float(t.split('mnt1_')[1].split('_')[0])
        mnt2 = float(t.split('mnt2_')[1].split('_')[0])
        mnt3= float(t.split('mnt3_')[1].split('_')[0])
        return [mnt1,mnt2,mnt3]
            
    elif task == 4:
        mnt1 = float(t.split('mnt1_')[1].split('_')[0])
        mnt2 = float(t.split('mnt2_')[1].split('_')[0])
        mnt3= float(t.split('mnt3_')[1].split('_')[0])
        mnt4 = float(t.split('mnt4_')[1].split('_')[0])
        return [mnt1,mnt2,mnt3,mnt4]
def get_items(trialnames, task):
    mnt1 = []
    mnt2 = []
    if task == 3:
        mnt3 = []
    if task == 4:
        mnt3 = []
        mnt4 = []
        
    for t in trialnames:
        #print(t)
        items = find_stimuli(t,task)
        mnt1.append(items[0])
        mnt2.append(items[1])
        if task == 3:
            mnt3.append(items[2])
        if task == 4:
            mnt3.append(items[3])
            mnt4.append(items[4])
    if task == 2:
        return mnt1, mnt2
    if task == 3:
        return mnt1, mnt2, mnt3
    if task == 4:
        return mnt1, mnt2, mnt3,mnt4


In [ ]:
#porportion of trials where there are all stimuli incldued - i.e. 1 item and 1 trial in memory; compare with 8 stripes vs. 2 stripes.

In [87]:
f = 'Y:/sir3_chunk/results/Ring/8Stripe/hybrid/decodedrew/contNoStimLocNoHidden/NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/lrate06/halfchunknochunk/sir3chunkhybridmodel1_RewThreshold5.5_decoded_Base.csv'
df = pd.read_csv(f, sep = '\t')

In [88]:
mnt1,mnt2,mnt3 = get_items(df['$TrialName'], 3)

In [89]:
stripes, stripename = get_stripes(df)

In [90]:
df_stripes = pd.DataFrame()
df_stripes['trialname'] = df['$TrialName']
for s in stripename: 
    df_stripes[s] = df[s]
df_stripes['mnt1'] = mnt1
df_stripes['mnt2'] = mnt2
df_stripes['mnt3'] = mnt3

In [91]:
df_stripes

,trialname,#stripe,#stripe,#stripe,#stripe,#stripe,#stripe,#stripe,#stripe,mnt1,mnt2,mnt3
0,Store2_84.62424715174222_mnt1_-1_mnt2_84.62424...,0.00,85.26,85.26,85.260,85.12,85.12,0.00,0.00,-1.000000,84.624247,-1.000000
1,Store3_312.7483857027013_mnt1_-1_mnt2_84.62424...,0.00,85.26,85.26,312.600,312.70,85.12,312.70,0.00,-1.000000,84.624247,312.748386
2,Recall3_312.7483857027013_mnt1_-1_mnt2_84.6242...,0.00,85.26,85.26,0.000,0.00,85.12,0.00,0.00,-1.000000,84.624247,-1.000000
3,Recall2_84.62424715174222_mnt1_-1_mnt2_-1_mnt3...,0.00,0.00,0.00,0.000,0.00,0.00,0.00,0.00,-1.000000,-1.000000,-1.000000
4,Store1_184.65368660065613_mnt1_184.65368660065...,180.20,180.20,0.00,0.000,0.00,0.00,0.00,185.60,184.653687,-1.000000,-1.000000
5,Store2_35.703238650595054_mnt1_184.65368660065...,183.50,34.74,34.74,34.740,29.19,29.19,0.00,185.60,184.653687,35.703239,-1.000000
6,Recall2_35.703238650595054_mnt1_184.6536866006...,183.50,0.00,0.00,0.000,0.00,0.00,0.00,185.60,184.653687,-1.000000,-1.000000
7,Store3_75.07242480163033_mnt1_184.653686600656...,183.50,0.00,0.00,75.100,75.27,0.00,75.27,185.60,184.653687,-1.000000,75.072425
8,Recall1_184.65368660065613_mnt1_-1_mnt2_-1_mnt...,0.00,0.00,0.00,75.370,75.27,0.00,75.27,0.00,-1.000000,-1.000000,75.072425
9,Store2_30.906947367871105_mnt1_-1_mnt2_30.9069...,0.00,47.50,47.50,47.500,22.41,22.41,75.27,0.00,-1.000000,30.906947,75.072425


In [30]:
df_stripes = pd.DataFrame()
df_stripes['trialname'] = df['$TrialName']
for s in stripename: 
    df_stripes[s] = df[s]
    

In [20]:
chunkfile = "Y:/sir3_chunk/results/Ring/8Stripe/hybrid/decodedrew/contNoStimLocNoHidden/NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/lrate06/halfchunknochunk/"
nochunkfile = "Y:/sir3/results/Ring/8Stripe/decodedrew/contNoStimLocNoHidden/NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/lrate06/halfchunknochunk/"
chunk_avg_stripes, nochunk_avg_stripes = avg_stripes(chunkfile= chunkfile, nochunkfile = nochunkfile )
print(sir4_avg_stripes)
print(sir4_chunk_avg_stripes)

all files: 64
> <ipython-input-19-7a5236add0ca>(30)avg_stripes()
-> stripes,stripename = get_stripes(pd.read_csv(f,sep = '\t'))


(Pdb)  f


'Y:/sir3_chunk/results/Ring/8Stripe/hybrid/decodedrew/contNoStimLocNoHidden/NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/lrate06/halfchunknochunk/sir3chunkhybridmodel0_RewThreshold5.5_decoded_Base.csv'


(Pdb)  quit()


BdbQuit: 

In [13]:
chunkfile = "Y:/sir3_chunk/results/Ring/8Stripe/hybrid/decodedrew/contNoStimLocNoHidden/NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/lrate06/halfchunknochunk/"
nochunkfile = "Y:/sir3/results/Ring/8Stripe/decodedrew/contNoStimLocNoHidden/NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/lrate06/halfchunknochunk/"
chunk_avg_stripes, nochunk_avg_stripes = avg_stripes(chunkfile= chunkfile, nochunkfile = nochunkfile )
print(sir4_avg_stripes)
print(sir4_chunk_avg_stripes)

all files: 64
all files: 64
7.53125
7.640625


In [7]:
base = "Y:/"
base2 = "results/Ring/8Stripe/"
experiment = "decodedrew/contNoStimLocNoHidden//NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/lrate06/halfchunknochunk/"
sir4_avg_stripes, sir4_chunk_avg_stripes = avg_stripes(base,base2,experiment)
print(sir4_avg_stripes)
print(sir4_chunk_avg_stripes)

all files: 64
all files: 64
7.53125
7.640625


In [9]:
base = "Y:/"
base2 = "results/Ring/8Stripe/"
experiment = "decodedrew/contNoStimLocNoHidden//NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/"
sir4_avg_stripes, sir4_chunk_avg_stripes = avg_stripes(base,base2,experiment)
print(sir4_avg_stripes)
print(sir4_chunk_avg_stripes)

all files: 64
all files: 64
7.9375
7.859375


In [14]:
base = "Y:/"
base2 = "results/Ring/8Stripe/"
experiment = "decodedrew/contNoStimLocNoHidden//NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/lrate06/"
sir3_avg_stripes, sir3_chunk_avg_stripes = avg_stripes(base,base2,experiment)
print(sir3_avg_stripes)
print(sir3_chunk_avg_stripes)

all files: 64
all files: 64
7.53125
7.59375


In [18]:
base = "Y:/"
base2 = "results/Ring/8Stripe/"
experiment = "decodedrew/contNoStimLocNoHidden//NoIgnore/nonresponse/OneR/defaults/VmNoise/VarNonFixed/"
sir2_avg_stripes, sir2_chunk_avg_stripes = avg_stripes(base,base2,experiment)
print(sir2_avg_stripes)
print(sir2_chunk_avg_stripes)

all files: 64
all files: 64
6.390625
6.109375


In [16]:
base = "Z:/leabra/examples/workingmemory/"
base2 = "results/Ring/8Stripe/"
experiment = "decodedrew/contNoStimLoc/NoIgnore/"
sir4_avg_stripes, sir4_chunk_avg_stripes = avg_stripes(base,base2,experiment)

all files: 56
all files: 53


In [17]:
print(sir4_avg_stripes)
print(sir4_chunk_avg_stripes)

7.464285714285714
6.981132075471698


In [12]:
base = "Z:/leabra/examples/workingmemory/"
base2 = "results/Ring/8Stripe/"
experiment = "decodedrew/contNoStimLoc/NoIgnore/sigma12/"
sir4_avg_stripes, sir4_chunk_avg_stripes = avg_stripes(base,base2,experiment)

all files: 54
all files: 49


In [13]:
print(sir4_avg_stripes)

7.444444444444445


In [15]:
print(sir4_chunk_avg_stripes)

6.673469387755102


In [34]:
full_files = [ex_file]
#full_files = getfiles(base)
p = Precision()
err = p.mult_models(full_files)
df_recall = p.dat.loc['True']
trials = len(df_recall)
mnt1 = []
mnt2 = []
mnt3= []
mnt4 = []
for i in range(trials):
    t = df_recall.iloc[i]['$TrialName']
    recall = t.split('Recall')[1].split('_')[0]
    if recall == '1':
        mnt1.append(float(t.split('_')[1]))
        mnt2.append(float(t.split('mnt2_')[1].split('_')[0]))
        mnt3.append(float(t.split('mnt3_')[1].split('_')[0]))
        mnt4.append(float(t.split('mnt4_')[1].split('_')[0]))
    if recall=='2':
        mnt1.append(float(t.split('mnt1_')[1].split('_')[0]))
        mnt2.append(float(t.split('_')[1]))
        mnt3.append(float(t.split('mnt3_')[1].split('_')[0]))
        mnt4.append(float(t.split('mnt4_')[1].split('_')[0]))
    if recall=='3':
        mnt1.append(float(t.split('mnt1_')[1].split('_')[0]))
        mnt2.append(float(t.split('mnt2_')[1].split('_')[0]))
        mnt3.append(float(t.split('_')[1]))
        mnt4.append(float(t.split('mnt4_')[1].split('_')[0]))
    if recall=='4':
        mnt1.append(float(t.split('mnt1_')[1].split('_')[0]))
        mnt2.append(float(t.split('mnt2_')[1].split('_')[0]))
        mnt3.append(float(t.split('mnt3_')[1].split('_')[0]))
        mnt4.append(float(t.split('_')[1]))